In [58]:
import numpy as np
import random
import math

def generer_matrices(t, difficulte=0.5, seed=1):
    """
    Génère deux matrices complètes :
    - MATRICES_P : distances positives symétriques
    - MATRICES_R : dangerosité (0 à 5), symétrique, influencée par 'difficulte'
    
    Paramètres :
    - t : taille de la matrice
    - difficulte : float entre 0 et 1 (0 = facile, 1 = très dangereux)
    - seed : pour reproductibilité
    
    Retour :
    - MATRICES_P (np.array)
    - MATRICES_R (np.array)
    """
    
    dist_min=10
    dist_max=80

    
    np.random.seed(seed)
    random.seed(seed)

    # -------------------------
    # MATRICE DES DISTANCES
    # -------------------------
    dist_matrix = np.random.randint(dist_min, dist_max + 1, size=(t, t))
    np.fill_diagonal(dist_matrix, 0)
    dist_matrix = (dist_matrix + dist_matrix.T) // 2  # symétrie

    # -------------------------
    # MATRICE DES COÛTS EN VIES
    # -------------------------
    MATRICES_R = np.zeros((t, t), dtype=int)

    for i in range(t):
        for j in range(i + 1, t):

            # probabilité d'avoir un coût élevé
            if random.random() < difficulte:
                # coût dangereux : 3 à 5
                cost = np.random.randint(3, 6)
            else:
                # coût faible : 0 à 2
                cost = np.random.randint(0, 3)

            MATRICES_R[i, j] = cost
            MATRICES_R[j, i] = cost

    return dist_matrix, MATRICES_R

P1 = generer_matrices(10)[0]
R1 = generer_matrices(10)[1]

P2 = generer_matrices(20)[0]
R2 = generer_matrices(20)[1]

P3 = generer_matrices(50)[0]
R3 = generer_matrices(50)[1]

MATRICES_P = [P1, P2, P3]
MATRICES_R = [R1, R2, R3]

# Affichage des matrices
for i in range(len(MATRICES_P)):
    print(f"Matrice P{i+1} (Distances) :\n{MATRICES_P[i]}\n")
    print(f"Matrice R{i+1} (Coûts en vies) :\n{MATRICES_R[i]}\n")


Matrice P1 (Distances) :
[[ 0 26 18 36 63 30 26 40 44 46]
 [26  0 47 39 37 28 52 76 26 18]
 [18 47  0 22 64 15 15 41 29 28]
 [36 39 22  0 41 68 50 11 62 32]
 [63 37 64 41  0 53 51 25 17 39]
 [30 28 15 68 53  0 41 64 51 20]
 [26 52 15 50 51 41  0 68 60 54]
 [40 76 41 11 25 64 68  0 49 43]
 [44 26 29 62 17 51 60 49  0 27]
 [46 18 28 32 39 20 54 43 27  0]]

Matrice R1 (Coûts en vies) :
[[0 3 3 5 5 5 3 4 5 5]
 [3 0 5 5 3 5 4 4 5 3]
 [3 5 0 5 3 5 4 4 4 5]
 [5 5 5 0 4 4 3 5 3 4]
 [5 3 3 4 0 3 4 4 3 4]
 [5 5 5 4 3 0 4 4 5 4]
 [3 4 4 3 4 4 0 3 3 5]
 [4 4 4 5 4 4 3 0 4 3]
 [5 5 4 3 3 5 3 4 0 3]
 [5 3 5 4 4 4 5 3 3 0]]

Matrice P2 (Distances) :
[[ 0 19 36 28 63 23 23 21 26 44 39 42 34 48 42 25 38 46 24 31]
 [19  0 53 38 48 37 17 54 30 50 54 26 46 75 43 53 46 32 50 43]
 [36 53  0 35 32 77 50 45 17 35 57 49 31 78 35 29 43 64 38 33]
 [28 38 35  0 35 55 62 66 51 40 50 65 33 16 27 47 50 39 35 37]
 [63 48 32 35  0 56 47 71 37 38 45 40 35 38 27 19 73 68 51 13]
 [23 37 77 55 56  0 45 35 50 33 45 41 49 2

# Métaheurestiques

In [40]:
import numpy as np
import random

def aco(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15, alpha=1, beta=2, rho=0.4, Q=50, num_ants=40, num_iterations=2000):
    """
    MATRICES_P : Matrice des distances (n x n)
    MATRICES_R : Matrice des coûts de vie (n x n)
    """
    n = MATRICES_P.shape[0]
    
    # Initialisation des phéromones et de l'heuristique
    pheromone = np.ones((n, n))
    # On évite la division par zéro
    heuristic = 1 / (MATRICES_P + 1e-10)
    np.fill_diagonal(heuristic, 0)

    # --- Fonctions internes ---
    def get_route_distance(route):
        return sum(MATRICES_P[route[i], route[i+1]] for i in range(len(route)-1))

    def evaluate_solution(routes, unvisited_left):
        distances = [get_route_distance(r) for r in routes]
        # Pénalité forte si des villes ne sont pas visitées
        if unvisited_left > 0:
            penalty = 10000 * unvisited_left
            return max(distances) + penalty, distances
        return max(distances), distances

    def choose_next(current, unvisited, life_left):
        # Filtrage par contrainte de vie
        feasible = [j for j in unvisited if MATRICES_R[current, j] <= life_left]
        if not feasible:
            return None

        weights = []
        total = 0
        for j in feasible:
            w = (pheromone[current, j] ** alpha) * (heuristic[current, j] ** beta)
            weights.append(w)
            total += w

        if total == 0:
            return random.choice(feasible)

        r = random.random() * total
        s = 0
        for j, w in zip(feasible, weights):
            s += w
            if s >= r:
                return j
        return feasible[-1]

    def build_solution():
        unvisited = set(range(1, n))
        routes = [[0] for _ in range(K)]
        life_left = [MAX_LIFE] * K
        vehicle = 0
        stuck_count = 0 

        while unvisited:
            current = routes[vehicle][-1]
            nxt = choose_next(current, list(unvisited), life_left[vehicle])

            if nxt is None:
                if routes[vehicle][-1] != 0:
                    routes[vehicle].append(0)
                
                vehicle = (vehicle + 1) % K
                stuck_count += 1
                if stuck_count >= K:
                    break
                continue

            stuck_count = 0
            routes[vehicle].append(nxt)
            life_left[vehicle] -= MATRICES_R[current, nxt]
            unvisited.remove(nxt)
            vehicle = (vehicle + 1) % K

        for v in range(K):
            if routes[v][-1] != 0:
                routes[v].append(0)

        return routes, len(unvisited)

    # --- Boucle ACO ---
    best_solution = None
    best_cost = float("inf")
    best_distances = []
    best_unvisited = n

    for it in range(num_iterations):
        solutions = []
        for _ in range(num_ants):
            routes, unvisited_left = build_solution()
            cost, distances = evaluate_solution(routes, unvisited_left)
            solutions.append((routes, cost, distances, unvisited_left))

            if cost < best_cost:
                best_cost = cost
                best_solution = routes
                best_distances = distances
                best_unvisited = unvisited_left

        # Évaporation
        pheromone *= (1 - rho)

        # Mise à jour des phéromones (Top 3 solutions valides)
        solutions.sort(key=lambda x: x[1])
        for routes, cost, distances, unvisited_left in solutions[:3]:
            if unvisited_left > 0:
                continue
            deposit = Q / (cost + 1e-10)
            for route in routes:
                for i in range(len(route)-1):
                    a, b = route[i], route[i+1]
                    pheromone[a, b] += deposit
                    pheromone[b, a] += deposit

    # --- Préparation du retour ---
    is_feasible = (best_unvisited == 0)
    
    # Transformation des routes en dictionnaire "1", "2", "3"...
    cycles_vaisseaux = {}
    if best_solution:
        for idx, r in enumerate(best_solution):
            cycles_vaisseaux[str(idx + 1)] = r
    else:
        # Cas où aucune solution n'a été construite du tout
        cycles_vaisseaux = {str(i+1): [0, 0] for i in range(K)}

    max_dist = max(best_distances) if best_distances else 0.0

    return is_feasible, cycles_vaisseaux, max_dist

In [51]:
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
def vns():
    return None
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------
# ----------------------------------------------------

In [41]:
import numpy as np
import random
import copy

def genetic_algorithm(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15, pop_size=50, generations=500, p_mutate=0.2):
    """
    MATRICES_P : Matrice des distances (n x n)
    MATRICES_R : Matrice des risques/perte de vie (n x n)
    MAX_LIFE   : Seuil de risque (int ou liste de taille K)
    """
    n = MATRICES_P.shape[0]
    
    # Gestion du MAX_LIFE (si c'est un int, on le transforme en liste pour chaque vaisseau)
    if isinstance(MAX_LIFE, (int, float)):
        R_max = [MAX_LIFE] * K
    else:
        R_max = MAX_LIFE

    # --- Fonctions internes ---
    def create_individual():
        nodes = list(range(1, n))
        random.shuffle(nodes)
        routes = [[] for _ in range(K)]
        for i, node in enumerate(nodes):
            routes[i % K].append(node)
        return [[0] + r + [0] for r in routes]

    def route_distance(route):
        return sum(MATRICES_P[route[i], route[i+1]] for i in range(len(route)-1))

    def route_risk(route):
        return sum(MATRICES_R[route[i], route[i+1]] for i in range(len(route)-1))

    def fitness(ind):
        distances = [route_distance(r) for r in ind]
        risks = [route_risk(r) for r in ind]
        penalty = 0
        for k in range(K):
            if risks[k] > R_max[k]:
                # Pénalité proportionnelle au dépassement
                penalty += 10000 * (risks[k] - R_max[k] + 1)
        return max(distances) + penalty

    def selection(pop):
        # Tournoi binaire
        i, j = random.sample(range(len(pop)), 2)
        return pop[i] if fitness(pop[i]) < fitness(pop[j]) else pop[j]

    def crossover(p1, p2):
        # Logique de croisement : mélange des noeuds visités
        child_nodes = []
        for r in p1:
            child_nodes += r[1:-1]
        random.shuffle(child_nodes)
        
        routes = [[] for _ in range(K)]
        for i, node in enumerate(child_nodes):
            routes[i % K].append(node)
        return [[0] + r + [0] for r in routes]

    def mutate(ind):
        new_ind = copy.deepcopy(ind)
        if random.random() < p_mutate:
            k1, k2 = random.sample(range(K), 2)
            # On déplace un noeud du vaisseau k1 vers k2
            if len(new_ind[k1]) > 3: # 3 car [0, node, 0]
                i = random.randint(1, len(new_ind[k1]) - 2)
                node = new_ind[k1].pop(i)
                j = random.randint(1, len(new_ind[k2]) - 1)
                new_ind[k2].insert(j, node)
        return new_ind

    # --- Boucle principale de l'AG ---
    population = [create_individual() for _ in range(pop_size)]
    best_ind = min(population, key=fitness)

    for _ in range(generations):
        new_population = []
        for _ in range(pop_size):
            parent1 = selection(population)
            parent2 = selection(population)
            
            enfant = crossover(parent1, parent2)
            enfant = mutate(enfant)
            new_population.append(enfant)
        
        population = new_population
        current_best = min(population, key=fitness)
        
        if fitness(current_best) < fitness(best_ind):
            best_ind = current_best

    # --- Préparation du retour ---
    # Calcul des risques finaux pour vérifier la faisabilité
    final_risks = [route_risk(r) for r in best_ind]
    is_feasible = all(final_risks[k] <= R_max[k] for k in range(K))
    
    # Construction du dictionnaire des cycles
    cycles_vaisseaux = {str(i + 1): route for i, route in enumerate(best_ind)}
    
    # Distance max réelle (sans la pénalité)
    max_dist = max(route_distance(r) for r in best_ind)

    return is_feasible, cycles_vaisseaux, max_dist


   

In [47]:
import numpy as np
import math
from copy import deepcopy

def tabu_search(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15, iterations=300, tenure=7):
    """
    Exécute une recherche tabou pour le problème de tournées de véhicules.
    
    Retourne:
    - is_feasible (bool)
    - cycles_vaisseaux (dict)
    - max_distance (float)
    """
    N = len(MATRICES_P)
    START = 0

    # --- Outils internes ---
    def get_route_distance(route):
        return sum(MATRICES_P[route[i]][route[i + 1]] for i in range(len(route) - 1))

    def get_route_lives(route):
        return sum(MATRICES_R[route[i]][route[i + 1]] for i in range(len(route) - 1))

    def check_feasibility(routes):
        visited = []
        for r in routes:
            if get_route_lives(r) > MAX_LIFE:
                return False
            visited.extend(r[1:-1])
        return sorted(visited) == list(range(1, N))

    def evaluate(routes, penalty=1000):
        distances = [get_route_distance(r) for r in routes]
        max_dist = max(distances)
        violation = 0
        for r in routes:
            lives = get_route_lives(r)
            if lives > MAX_LIFE:
                violation += (lives - MAX_LIFE)
        return max_dist + penalty * violation

    def get_neighbors(routes):
        neigh = []
        for a in range(len(routes)):
            for b in range(len(routes)):
                if a == b: continue
                ra, rb = routes[a], routes[b]
                for i in range(1, len(ra) - 1):
                    node = ra[i]
                    new_ra = ra[:i] + ra[i+1:]
                    for j in range(1, len(rb)):
                        new_rb = rb[:j] + [node] + rb[j:]
                        new_sol = [list(r) for r in routes] # Copie rapide
                        new_sol[a] = new_ra
                        new_sol[b] = new_rb
                        neigh.append((new_sol, (node, a, b)))
        return neigh

    # --- Initialisation ---
    customers = list(range(1, N))
    current_sol = [[START] for _ in range(K)]
    for i, c in enumerate(customers):
        current_sol[i % K].append(c)
    for i in range(K):
        current_sol[i].append(START)

    best_sol = deepcopy(current_sol)
    best_score = evaluate(best_sol)
    tabu_list = {} # move -> iteration d'expiration

    # --- Boucle principale ---
    for it in range(iterations):
        cand_best, cand_move, cand_score = None, None, math.inf

        for sol, move in get_neighbors(current_sol):
            score = evaluate(sol)
            
            # Critère d'aspiration : on ignore le tabou si on bat le record absolu
            is_tabu = move in tabu_list and tabu_list[move] > it
            if is_tabu and score >= best_score:
                continue

            if score < cand_score:
                cand_score, cand_best, cand_move = score, sol, move

        if cand_best is None:
            break

        current_sol = cand_best
        tabu_list[cand_move] = it + tenure

        # Mise à jour du meilleur score global si la solution est réalisable
        if cand_score < best_score and check_feasibility(current_sol):
            best_sol = deepcopy(current_sol)
            best_score = cand_score

    # --- Formatage de la sortie ---
    is_feasible = check_feasibility(best_sol)
    cycles_vaisseaux = {str(i + 1): route for i, route in enumerate(best_sol)}
    
    # On recalcule la distance max réelle sans pénalité
    max_distance = max(get_route_distance(r) for r in best_sol)

    return is_feasible, cycles_vaisseaux, max_distance

# MILP

In [54]:
import pulp

def solve_milp_exact(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15):
    """
    Version corrigée du solveur exact MILP.
    """
    N = len(MATRICES_P)
    START = 0
    clients = [i for i in range(N) if i != START]
    
    model = pulp.LpProblem("MinMax_kTSP_Exact", pulp.LpMinimize)

    # --- Variables ---
    # On crée toutes les combinaisons i, j pour x
    x = pulp.LpVariable.dicts("x", (range(K), range(N), range(N)), cat="Binary")
    y = pulp.LpVariable.dicts("y", (range(K), range(N)), cat="Binary")
    u = pulp.LpVariable.dicts("u", (range(K), clients), lowBound=1, upBound=len(clients))
    z = pulp.LpVariable("z", lowBound=0)

    model += z

    # --- Contraintes ---
    for i in clients:
        model += pulp.lpSum(y[v][i] for v in range(K)) == 1

    for v in range(K):
        for i in clients:
            model += pulp.lpSum(x[v][j][i] for j in range(N) if j != i) == y[v][i]
            model += pulp.lpSum(x[v][i][j] for j in range(N) if j != i) == y[v][i]

    for v in range(K):
        # Départ et retour obligatoires si le véhicule est utilisé
        model += pulp.lpSum(x[v][START][j] for j in clients) <= 1
        model += pulp.lpSum(x[v][j][START] for j in clients) == pulp.lpSum(x[v][START][j] for j in clients)

    for v in range(K):
        model += pulp.lpSum(MATRICES_P[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= z

    for v in range(K):
        model += pulp.lpSum(MATRICES_R[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= MAX_LIFE

    # MTZ Anti-sous-tours
    M = len(clients)
    for v in range(K):
        for i in clients:
            for j in clients:
                if i != j:
                    model += u[v][i] - u[v][j] + M * x[v][i][j] <= M - 1

    # Résolution
    model.solve(pulp.PULP_CBC_CMD(msg=False))
    
    is_feasible = pulp.LpStatus[model.status] == 'Optimal'
    max_distance = pulp.value(z) if is_feasible and pulp.value(z) is not None else 0.0
    
    # --- Reconstruction des cycles (CORRIGÉE) ---
    cycles_vaisseaux = {}
    if is_feasible:
        for v in range(K):
            route = [START]
            curr = START
            # On limite la boucle pour éviter les tours infinis au cas où
            for _ in range(N):
                next_node = None
                for j in range(N):
                    if curr != j:
                        # pulp.value peut renvoyer None si la variable n'est pas utilisée
                        val = pulp.value(x[v][curr][j])
                        if val is not None and val > 0.5:
                            next_node = j
                            break
                
                if next_node is None or next_node == START:
                    break
                
                route.append(next_node)
                curr = next_node
            
            route.append(START)
            cycles_vaisseaux[str(v + 1)] = route
    else:
        cycles_vaisseaux = {str(v + 1): [0, 0] for v in range(K)}

    return is_feasible, cycles_vaisseaux, max_distance

In [56]:
def solve_strong_lp(MATRICES_P, MATRICES_R, K=3, MAX_LIFE=15):
    """
    Borne inférieure via relaxation linéaire.
    """
    N = len(MATRICES_P)
    model = pulp.LpProblem("Strong_LP", pulp.LpMinimize)

    # Variables continues entre 0 et 1 (Relaxation)
    x = pulp.LpVariable.dicts("x", (range(K), range(N), range(N)), 0, 1, cat="Continuous")
    y = pulp.LpVariable.dicts("y", (range(K), range(N)), 0, 1, cat="Continuous")
    z = pulp.LpVariable("z", lowBound=0)

    model += z

    # Chaque client visité au total une fois (somme des fractions = 1)
    for i in range(1, N):
        model += pulp.lpSum(y[v][i] for v in range(K)) == 1

    # Flux
    for v in range(K):
        for i in range(1, N):
            model += pulp.lpSum(x[v][i][j] for j in range(N) if j != i) == y[v][i]
            model += pulp.lpSum(x[v][j][i] for j in range(N) if j != i) == y[v][i]

    # Dépôt
    for v in range(K):
        model += pulp.lpSum(x[v][0][j] for j in range(1, N)) <= 1
        model += pulp.lpSum(x[v][j][0] for j in range(1, N)) <= 1

    # Distance max
    for v in range(K):
        model += pulp.lpSum(MATRICES_P[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= z

    # Vies
    for v in range(K):
        model += pulp.lpSum(MATRICES_R[i][j] * x[v][i][j] for i in range(N) for j in range(N) if i != j) <= MAX_LIFE

    model.solve(pulp.PULP_CBC_CMD(msg=False))

    is_feasible = pulp.LpStatus[model.status] == 'Optimal'
    
    # Pas de cycles réels en LP relaxation car les routes sont fractionnaires
    cycles_vaisseaux = {str(v + 1): [] for v in range(K)}
    max_dist = pulp.value(model.objective) if is_feasible else 0.0

    return is_feasible, cycles_vaisseaux, max_dist

In [59]:
NB_VEHICULE=3
MAX_LIFE=15

def affichage_propre(faisable, cycles, max_distance):
    print(f"faisabilite: {faisable}")
    if len(cycles):
        for idx, cycle in enumerate(cycles):
            print(f"Véhicule {idx} : {cycle}")
    print(f"Distance max: {max_distance}")

print(aco(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, alpha=1, beta=2, rho=0.4, Q=50, num_ants=40, num_iterations=200))
# print(vns(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, **kwargs ))      # A Completer
print(genetic_algorithm(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, pop_size=50, generations=50, p_mutate=0.2))
print(tabu_search(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE, iterations=300, tenure=7))

#print(solve_milp_exact(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE)) #Attention il prend du temps lui !
print(solve_strong_lp(P1, R1, K=NB_VEHICULE, MAX_LIFE=MAX_LIFE))


(True, {'1': [0, 5, 9, 3, 0], '2': [0, 1, 4, 8, 0], '3': [0, 6, 2, 7, 0]}, 124)
(True, {'1': [0, 1, 9, 5, 0], '2': [0, 7, 4, 6, 0], '3': [0, 2, 8, 3, 0]}, 145)
(True, {'1': [0, 2, 8, 7, 0], '2': [0, 1, 4, 5, 0], '3': [0, 6, 3, 9, 0]}, 154)
(True, {'1': [0, 2, 8, 4, 0], '2': [0, 5, 9, 1, 0], '3': [0, 6, 3, 7, 0]}, 127.0)
(True, {'1': [], '2': [], '3': []}, 50.666667)
